# CNeuroMod GLM — Results

Visualizes session-level and subject-level contrast maps produced by `invoke run-glm` and `invoke run-subject`.  
Reads from `OUTPUT_DATA_DIR` (set by invoke when executing via `invoke run-notebooks`).

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from nilearn import plotting, image

OUTPUT_DATA_DIR = Path(os.environ.get('OUTPUT_DATA_DIR', 'output_data'))
FIGURES_DIR = OUTPUT_DATA_DIR / 'figures_glm'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['sub-01', 'sub-02', 'sub-03', 'sub-05']
DIFF_CONTRASTS = {
    'listening': 'int-degr',
    'aliceFr':   'int-degr',
    'aliceEn':   'int-degr',
    'reading':   'word-nonword',
}
THRESHOLD = 3.0

print(f'Output data dir: {OUTPUT_DATA_DIR}')
print(f'Figures dir:     {FIGURES_DIR}')

Output data dir: /scratch/ibilgin/cneuromod.glm/output_data
Figures dir:     /scratch/ibilgin/cneuromod.glm/output_data/figures_glm


In [2]:
# --- Subject-level contrast maps (one panel per task) ---
for subject in SUBJECTS:
    tasks_found = []
    imgs = []
    for task, contrast in DIFF_CONTRASTS.items():
        z_map = (
            OUTPUT_DATA_DIR / subject / 'subject_level' / task /
            f'{subject}_task-{task}_contrast-{contrast}_stat-z.nii.gz'
        )
        if z_map.exists():
            tasks_found.append((task, contrast, z_map))

    if not tasks_found:
        print(f'{subject}: no subject-level maps found — run invoke run-subject first')
        continue

    fig, axes = plt.subplots(len(tasks_found), 1, figsize=(18, 5 * len(tasks_found)))
    if len(tasks_found) == 1:
        axes = [axes]
    fig.suptitle(f'{subject} — Subject-Level Fixed-Effects', fontsize=14, fontweight='bold')

    for ax, (task, contrast, z_map) in zip(axes, tasks_found):
        img = image.load_img(str(z_map))
        plotting.plot_stat_map(
            img,
            threshold=THRESHOLD,
            colorbar=True,
            cmap='cold_hot',
            display_mode='z',
            cut_coords=[-18, 3, 27, 47, 55],
            axes=ax,
            title=f'{task} | {contrast}',
        )

    out_png = FIGURES_DIR / f'{subject}_subject_level_summary.png'
    plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_png.name}')

sub-01: no subject-level maps found — run invoke run-subject first
sub-02: no subject-level maps found — run invoke run-subject first
sub-03: no subject-level maps found — run invoke run-subject first
sub-05: no subject-level maps found — run invoke run-subject first


In [3]:
print('Done. Figures saved to', FIGURES_DIR)

Done. Figures saved to /scratch/ibilgin/cneuromod.glm/output_data/figures_glm
